In [1]:
import tensorflow as tf

In [2]:
# 1. 请来“绘画大师”（载入预训练模型，这里选择轻量级的MobileNetV2）
# include_top = False: 不要它原来识别 1000 种物质的分类头（嘴巴）
# weights = 'imagenet':加载它在 ImageNet 千万张图片上练出的权重（基本功）
base_model = tf.keras.applications.MobileNetV2(input_shape=(160,160,3),
                                              include_top=False,
                                              weights='imagenet')



9406464/9406464 [==============================] - 23s 2us/step


In [4]:
# 2. 冻结卷积基（核心操作！）
#这一步保证了我们在初期训练时，不会破话它原本提取边缘和纹理的能力
base_model.trainable= False

# 3. 组装新模型（接上我们自己定制的输出层）
model =  tf.keras.Sequential([
    base_model,                          #第一层：冻结的预训练大师
    tf.keras.layers.GlobalAveragePooling2D(),   # 第二层：把大师提取的复杂特征压平
    tf.keras.layers.Dense(10, activation='softmax')   #第三层：假设我们还是要分10类
])

In [5]:
# 4. 编译模型（注意这里只训练最后加的 Dense 层）
model.compile(optimizer='adam',
             loss='sparse_categorical_crossentropy',
             metrics=['accuracy'])

#查看模型结构：你会发现总参数有两百多万，但“可训练参数”只有几千个
model.summary()


Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 mobilenetv2_1.00_160 (Func  (None, 5, 5, 1280)        2257984   
 tional)                                                         
                                                                 
 global_average_pooling2d_1  (None, 1280)              0         
  (GlobalAveragePooling2D)                                       
                                                                 
 dense_1 (Dense)             (None, 10)                12810     
                                                                 
Total params: 2270794 (8.66 MB)
Trainable params: 12810 (50.04 KB)
Non-trainable params: 2257984 (8.61 MB)
_________________________________________________________________
